# TensorBoard可视化训练过程 — PyTorch版

> **PyTorch等效版本** | 原始TensorFlow/Keras版本: [set_log_position.ipynb](./set_log_position.ipynb)

本教程介绍如何使用 **PyTorch + TensorBoard** 监控和可视化神经网络的训练过程。

TensorBoard 本质上是一个 **框架无关(framework-agnostic)** 的可视化工具，它读取的是事件文件(event files)，
与生成这些文件的深度学习框架无关。PyTorch 通过 `torch.utils.tensorboard.SummaryWriter` 提供了一等公民级别的支持。

## 学习目标

1. 理解TensorBoard在PyTorch中的基本功能
2. 掌握 `SummaryWriter` 日志目录的设置方法
3. 学会使用 `add_scalar`、`add_histogram`、`add_graph`、`add_image` 等方法
4. 了解TF与PyTorch中TensorBoard用法的对照关系

## TensorBoard功能概览

| 功能 | 描述 | PyTorch方法 | 用途 |
|------|------|-------------|------|
| Scalars | 标量指标曲线 | `add_scalar` | 监控损失和指标 |
| Graphs | 计算图可视化 | `add_graph` | 理解模型结构 |
| Distributions | 权重分布 | `add_histogram` | 检测梯度问题 |
| Histograms | 直方图 | `add_histogram` | 分析权重变化 |
| Images | 图像数据 | `add_image` | 可视化输入/输出 |
| Text | 文本数据 | `add_text` | 记录配置/超参数 |

## 1. 环境配置与数据准备

In [ ]:
import os
import time

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.tensorboard import SummaryWriter

# 设置随机种子 / Set random seed
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print(f"PyTorch版本: {torch.__version__}")
print(f"CUDA可用: {torch.cuda.is_available()}")

# 设备选择 / Device selection
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")

In [ ]:
# 加载California Housing数据集 / Load California Housing dataset
housing = fetch_california_housing()

# 划分数据集 / Split dataset
X_train_full, X_test, y_train_full, y_test = train_test_split(
    housing.data, housing.target, test_size=0.2, random_state=RANDOM_SEED
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=RANDOM_SEED
)

# 标准化 / Standardize
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_valid = scaler.transform(X_valid)
X_test = scaler.transform(X_test)

print(f"训练集: {X_train.shape}")
print(f"验证集: {X_valid.shape}")
print(f"测试集: {X_test.shape}")

# 转换为PyTorch张量 / Convert to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.FloatTensor(y_train).unsqueeze(1)
X_valid_tensor = torch.FloatTensor(X_valid)
y_valid_tensor = torch.FloatTensor(y_valid).unsqueeze(1)
X_test_tensor = torch.FloatTensor(X_test)
y_test_tensor = torch.FloatTensor(y_test).unsqueeze(1)

# 创建DataLoader / Create DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

valid_dataset = TensorDataset(X_valid_tensor, y_valid_tensor)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False)

test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

## 2. 设置日志目录与SummaryWriter

PyTorch 中使用 `torch.utils.tensorboard.SummaryWriter` 来写入 TensorBoard 事件文件。
为每次训练创建唯一的日志目录，便于对比不同实验。

In [ ]:
# 设置根日志目录 / Set root log directory
root_logdir = os.path.join(os.curdir, "my_logs_pytorch")

def get_run_logdir(name=None):
    """
    生成带时间戳的日志目录路径
    Generate a timestamped log directory path.

    Parameters / 参数:
    -----------
    name : str, optional
        实验名称，用于区分不同的实验 / Experiment name for distinguishing runs

    Returns / 返回:
    --------
    str : 日志目录路径 / Log directory path
    """
    run_id = time.strftime("run_%Y_%m_%d-%H_%M_%S")
    if name:
        run_id = f"{name}_{run_id}"
    return os.path.join(root_logdir, run_id)

# 创建日志目录 / Create log directory
run_logdir = get_run_logdir("baseline")
print(f"日志目录: {run_logdir}")

In [ ]:
# 创建SummaryWriter / Create SummaryWriter
writer = SummaryWriter(log_dir=run_logdir)

print(f"SummaryWriter已创建，日志将保存到: {run_logdir}")
print("\nSummaryWriter主要方法:")
print("  add_scalar(tag, scalar_value, step)   - 记录标量 / Log scalar")
print("  add_scalars(main_tag, tag_dict, step) - 记录多个标量 / Log multiple scalars")
print("  add_histogram(tag, values, step)      - 记录直方图 / Log histogram")
print("  add_graph(model, input)               - 记录模型图 / Log model graph")
print("  add_image(tag, img_tensor, step)      - 记录图像 / Log image")
print("  add_text(tag, text_string, step)      - 记录文本 / Log text")
print("  flush()                               - 刷新到磁盘 / Flush to disk")
print("  close()                               - 关闭writer / Close writer")

## 3. 定义模型与训练函数

创建PyTorch回归模型，并封装训练循环以便在训练过程中记录TensorBoard指标。

In [ ]:
class RegressionModel(nn.Module):
    """
    多层感知机回归模型
    Multi-layer perceptron regression model.

    Parameters / 参数:
    -----------
    input_dim : int
        输入特征维度 / Input feature dimension
    n_hidden : int
        隐藏层数量 / Number of hidden layers
    n_neurons : int
        每层神经元数量 / Number of neurons per hidden layer
    activation : str
        激活函数名称 / Activation function name
    """

    def __init__(self, input_dim=8, n_hidden=2, n_neurons=30, activation='relu'):
        super().__init__()

        # 激活函数映射 / Activation function mapping
        activations = {
            'relu': nn.ReLU(),
            'tanh': nn.Tanh(),
            'sigmoid': nn.Sigmoid(),
            'elu': nn.ELU(),
        }
        act_fn = activations.get(activation, nn.ReLU())

        # 构建层列表 / Build layer list
        layers = [nn.Linear(input_dim, n_neurons), act_fn]
        for _ in range(n_hidden - 1):
            layers.append(nn.Linear(n_neurons, n_neurons))
            layers.append(act_fn)
        layers.append(nn.Linear(n_neurons, 1))

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        """前向传播 / Forward pass."""
        return self.network(x)


def create_model(input_dim=8, n_hidden=2, n_neurons=30, activation='relu', lr=0.01):
    """
    创建回归模型和优化器
    Create regression model and optimizer.

    Parameters / 参数:
    -----------
    input_dim : int
        输入特征维度 / Input feature dimension
    n_hidden : int
        隐藏层数量 / Number of hidden layers
    n_neurons : int
        每层神经元数量 / Number of neurons per hidden layer
    activation : str
        激活函数名称 / Activation function name
    lr : float
        学习率 / Learning rate

    Returns / 返回:
    --------
    tuple : (model, optimizer, criterion)
    """
    model = RegressionModel(
        input_dim=input_dim,
        n_hidden=n_hidden,
        n_neurons=n_neurons,
        activation=activation
    ).to(device)

    optimizer = optim.SGD(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    return model, optimizer, criterion

## 4. 记录标量 (Scalars) — 损失、精度、学习率

使用 `add_scalar` 记录训练过程中的标量指标，如损失(loss)、MAE和学习率。

In [ ]:
def train_with_tensorboard(model, optimizer, criterion, train_loader, valid_loader,
                           writer, epochs=30, log_histograms=True, log_graph=True):
    """
    使用TensorBoard记录的训练循环
    Training loop with TensorBoard logging.

    Parameters / 参数:
    -----------
    model : nn.Module
        PyTorch模型 / PyTorch model
    optimizer : optim.Optimizer
        优化器 / Optimizer
    criterion : nn.Module
        损失函数 / Loss function
    train_loader : DataLoader
        训练数据加载器 / Training data loader
    valid_loader : DataLoader
        验证数据加载器 / Validation data loader
    writer : SummaryWriter
        TensorBoard写入器 / TensorBoard writer
    epochs : int
        训练轮数 / Number of epochs
    log_histograms : bool
        是否记录权重直方图 / Whether to log weight histograms
    log_graph : bool
        是否记录模型图 / Whether to log model graph

    Returns / 返回:
    --------
    dict : 训练历史记录 / Training history
    """
    history = {'train_loss': [], 'val_loss': [], 'train_mae': [], 'val_mae': []}
    mae_criterion = nn.L1Loss()

    # 记录模型计算图 / Log model graph (only once)
    if log_graph:
        try:
            sample_input = next(iter(train_loader))[0][:1].to(device)
            writer.add_graph(model, sample_input)
            print("模型图已记录到TensorBoard / Model graph logged to TensorBoard")
        except Exception as e:
            print(f"无法记录模型图: {e}")

    # 记录模型配置文本 / Log model config as text
    model_info = str(model)
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    writer.add_text('Model/Architecture', f'```\n{model_info}\n```', 0)
    writer.add_text('Model/Parameters',
                    f'Total: {total_params:,} | Trainable: {trainable_params:,}', 0)

    for epoch in range(epochs):
        # === 训练阶段 / Training phase ===
        model.train()
        train_loss_sum = 0.0
        train_mae_sum = 0.0
        train_batches = 0

        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)

            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item() * batch_X.size(0)
            train_mae_sum += mae_criterion(outputs, batch_y).item() * batch_X.size(0)
            train_batches += batch_X.size(0)

        train_loss = train_loss_sum / train_batches
        train_mae = train_mae_sum / train_batches

        # === 验证阶段 / Validation phase ===
        model.eval()
        val_loss_sum = 0.0
        val_mae_sum = 0.0
        val_batches = 0

        with torch.no_grad():
            for batch_X, batch_y in valid_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                val_loss_sum += loss.item() * batch_X.size(0)
                val_mae_sum += mae_criterion(outputs, batch_y).item() * batch_X.size(0)
                val_batches += batch_X.size(0)

        val_loss = val_loss_sum / val_batches
        val_mae = val_mae_sum / val_batches

        # === 记录标量到TensorBoard / Log scalars to TensorBoard ===
        writer.add_scalar('Loss/train', train_loss, epoch)
        writer.add_scalar('Loss/val', val_loss, epoch)
        writer.add_scalar('MAE/train', train_mae, epoch)
        writer.add_scalar('MAE/val', val_mae, epoch)

        # 使用add_scalars对比训练和验证损失 / Compare train vs val using add_scalars
        writer.add_scalars('Loss/train_vs_val', {
            'train': train_loss,
            'val': val_loss
        }, epoch)
        writer.add_scalars('MAE/train_vs_val', {
            'train': train_mae,
            'val': val_mae
        }, epoch)

        # === 记录学习率 / Log learning rate ===
        current_lr = optimizer.param_groups[0]['lr']
        writer.add_scalar('Learning_Rate/SGD', current_lr, epoch)

        # === 记录过拟合比率 / Log overfitting ratio ===
        if val_loss > 0:
            overfit_ratio = train_loss / val_loss
            writer.add_scalar('Diagnostics/overfit_ratio', overfit_ratio, epoch)

        # === 记录权重直方图 / Log weight histograms ===
        if log_histograms:
            for name, param in model.named_parameters():
                writer.add_histogram(f'Weights/{name}', param, epoch)
                if param.grad is not None:
                    writer.add_histogram(f'Gradients/{name}', param.grad, epoch)

        # 保存历史 / Save history
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_mae'].append(train_mae)
        history['val_mae'].append(val_mae)

        # 打印进度 / Print progress
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:3d}/{epochs} | "
                  f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
                  f"Train MAE: {train_mae:.4f} | Val MAE: {val_mae:.4f}")

    writer.flush()
    return history

In [ ]:
# 创建模型并训练 / Create model and train
model, optimizer, criterion = create_model(n_hidden=2, n_neurons=30)

print(model)
print(f"\n总参数量: {sum(p.numel() for p in model.parameters()):,}")

# 使用SummaryWriter训练 / Train with SummaryWriter
baseline_logdir = get_run_logdir("baseline")
baseline_writer = SummaryWriter(log_dir=baseline_logdir)

history = train_with_tensorboard(
    model, optimizer, criterion,
    train_loader, valid_loader,
    baseline_writer,
    epochs=30,
    log_histograms=True,
    log_graph=True
)

baseline_writer.close()
print(f"\n日志已保存到: {baseline_logdir}")

## 5. 记录直方图 (Histograms) — 权重与梯度

`add_histogram` 用于记录张量的分布情况，这在监控权重和梯度变化时非常有用。
上面的训练循环已经展示了基本的直方图记录。这里我们深入探讨直方图的意义。

In [ ]:
# 详细记录权重和梯度统计 / Detailed weight and gradient statistics
def log_weight_statistics(model, writer, step):
    """
    记录模型权重的详细统计信息
    Log detailed weight statistics of the model.

    Parameters / 参数:
    -----------
    model : nn.Module
        PyTorch模型 / PyTorch model
    writer : SummaryWriter
        TensorBoard写入器 / TensorBoard writer
    step : int
        当前步骤 / Current step
    """
    for name, param in model.named_parameters():
        # 直方图 / Histogram
        writer.add_histogram(f'Params/{name}', param, step)

        # 标量统计 / Scalar statistics
        writer.add_scalar(f'Param_Stats/{name}_mean', param.data.mean(), step)
        writer.add_scalar(f'Param_Stats/{name}_std', param.data.std(), step)
        writer.add_scalar(f'Param_Stats/{name}_max', param.data.max(), step)
        writer.add_scalar(f'Param_Stats/{name}_min', param.data.min(), step)

        if param.grad is not None:
            writer.add_histogram(f'Gradients/{name}', param.grad, step)
            writer.add_scalar(f'Grad_Stats/{name}_norm', param.grad.norm(), step)
            writer.add_scalar(f'Grad_Stats/{name}_mean', param.grad.mean(), step)

# 创建新模型来演示详细的直方图记录 / Create a new model for detailed histogram demo
hist_model, hist_optimizer, hist_criterion = create_model(n_hidden=2, n_neurons=50)

hist_logdir = get_run_logdir("histograms")
hist_writer = SummaryWriter(log_dir=hist_logdir)

# 手动训练几个epoch来展示直方图 / Train a few epochs manually to show histograms
for epoch in range(15):
    hist_model.train()
    epoch_loss = 0.0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        hist_optimizer.zero_grad()
        outputs = hist_model(batch_X)
        loss = hist_criterion(outputs, batch_y)
        loss.backward()
        hist_optimizer.step()
        epoch_loss += loss.item()

    # 记录权重统计 / Log weight statistics each epoch
    log_weight_statistics(hist_model, hist_writer, epoch)
    hist_writer.add_scalar('Loss/train', epoch_loss / len(train_loader), epoch)

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/15, Loss: {epoch_loss/len(train_loader):.4f}")

hist_writer.close()
print(f"\n直方图日志保存到: {hist_logdir}")
print("\n在TensorBoard中，你可以在 Distributions 和 Histograms 标签页查看权重分布变化。")
print("权重的分布随训练的变化可以帮助你诊断:")
print("  - 梯度消失: 权重几乎不变化")
print("  - 梯度爆炸: 权重分布急剧扩大")
print("  - 健康训练: 权重分布缓慢变化并趋于稳定")

## 6. 记录模型图 (Graph)

使用 `add_graph` 记录模型的计算图。TensorBoard 会在 Graphs 标签页中显示模型的结构。
这对于理解模型架构和调试非常有帮助。

In [ ]:
# 记录模型图 / Log model graph
graph_model, _, _ = create_model(n_hidden=2, n_neurons=30)

graph_logdir = get_run_logdir("graph")
graph_writer = SummaryWriter(log_dir=graph_logdir)

# 创建示例输入 / Create sample input
sample_input = torch.randn(1, 8).to(device)

# 记录模型图 / Log model graph
graph_writer.add_graph(graph_model, sample_input)

graph_writer.close()
print(f"模型图已记录到: {graph_logdir}")
print("\n注意: add_graph 的第二个参数是模型的输入张量，用于追踪前向传播。")
print("PyTorch使用torch.jit.trace来追踪模型，因此输入的形状需要与实际训练时一致。")

# 对比：更复杂模型的图 / Compare: graph of a more complex model
complex_model = RegressionModel(input_dim=8, n_hidden=4, n_neurons=64, activation='elu').to(device)
complex_logdir = get_run_logdir("complex_graph")
complex_writer = SummaryWriter(log_dir=complex_logdir)
complex_writer.add_graph(complex_model, sample_input)
complex_writer.close()
print(f"\n复杂模型图已记录到: {complex_logdir}")

## 7. 记录图像 (Images)

使用 `add_image` 记录图像数据。在PyTorch中，图像张量的形状为 `(C, H, W)`。
这里我们使用Fashion-MNIST数据集来演示图像记录功能。

In [ ]:
# 使用Fashion-MNIST演示图像记录 / Use Fashion-MNIST for image logging demo
from torchvision import datasets, transforms

# 下载Fashion-MNIST / Download Fashion-MNIST
fashion_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,))  # Fashion-MNIST均值和标准差
])

fashion_train = datasets.FashionMNIST(
    root='./data', train=True, download=True, transform=fashion_transform
)

class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print(f"Fashion-MNIST训练集大小: {len(fashion_train)}")
print(f"类别: {class_names}")

In [ ]:
# 记录Fashion-MNIST图像到TensorBoard / Log Fashion-MNIST images to TensorBoard
from torchvision.utils import make_grid

img_logdir = get_run_logdir("fashion_mnist_images")
img_writer = SummaryWriter(log_dir=img_logdir)

# 记录单张图像 / Log single image
img, label = fashion_train[0]
# add_image需要(C, H, W)格式，ToTensor()已经提供了正确格式
img_writer.add_image('FashionMNIST/single_sample', img, 0)
img_writer.add_text('FashionMNIST/single_label', f'Class: {class_names[label]}', 0)

# 收集每个类别的一个样本 / Collect one sample per class
class_images = []
class_labels_found = set()
for img_tensor, lbl in fashion_train:
    if lbl not in class_labels_found:
        class_images.append(img_tensor)
        class_labels_found.add(lbl)
    if len(class_labels_found) == 10:
        break

# 按类别顺序排列 / Sort by class order
class_images_sorted = [class_images[lbl] for lbl in range(10)]

# 创建图像网格 / Create image grid
grid = make_grid(class_images_sorted, nrow=5, normalize=True)
img_writer.add_image('FashionMNIST/class_overview', grid, 0)

# 记录带标签的文本 / Log text with class labels
label_text = '\n'.join([f'{i}: {class_names[i]}' for i in range(10)])
img_writer.add_text('FashionMNIST/class_labels', label_text, 0)

img_writer.close()
print(f"Fashion-MNIST图像已记录到: {img_logdir}")
print("\n在TensorBoard的Images标签页中可以查看这些图像。")
print("注意: add_image接受的张量形状为(C, H, W)，而matplotlib使用(H, W, C)。")
print("make_grid会自动处理多个图像的拼接。")

In [ ]:
# 使用Fashion-MNIST训练一个简单分类器，同时记录到TensorBoard
# Train a simple classifier on Fashion-MNIST with TensorBoard logging

fashion_loader = DataLoader(fashion_train, batch_size=64, shuffle=True)

fashion_test = datasets.FashionMNIST(
    root='./data', train=False, download=True, transform=fashion_transform
)
fashion_test_loader = DataLoader(fashion_test, batch_size=64, shuffle=False)

# 简单CNN分类器 / Simple CNN classifier
class FashionCNN(nn.Module):
    """
    Fashion-MNIST分类CNN
    CNN classifier for Fashion-MNIST.
    """
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 64),
            nn.ReLU(),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        """前向传播 / Forward pass."""
        x = self.features(x)
        x = self.classifier(x)
        return x


fashion_model = FashionCNN().to(device)
fashion_optimizer = optim.Adam(fashion_model.parameters(), lr=0.001)
fashion_criterion = nn.CrossEntropyLoss()

fashion_logdir = get_run_logdir("fashion_cnn")
fashion_writer = SummaryWriter(log_dir=fashion_logdir)

# 记录模型图 / Log model graph
fashion_sample = torch.randn(1, 1, 28, 28).to(device)
fashion_writer.add_graph(fashion_model, fashion_sample)

# 训练 / Train
for epoch in range(5):
    fashion_model.train()
    correct = 0
    total = 0
    running_loss = 0.0

    for batch_X, batch_y in fashion_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        fashion_optimizer.zero_grad()
        outputs = fashion_model(batch_X)
        loss = fashion_criterion(outputs, batch_y)
        loss.backward()
        fashion_optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += batch_y.size(0)
        correct += predicted.eq(batch_y).sum().item()

    train_loss = running_loss / len(fashion_loader)
    train_acc = correct / total

    # 验证 / Validate
    fashion_model.eval()
    val_correct = 0
    val_total = 0
    val_loss_sum = 0.0
    with torch.no_grad():
        for batch_X, batch_y in fashion_test_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = fashion_model(batch_X)
            loss = fashion_criterion(outputs, batch_y)
            val_loss_sum += loss.item()
            _, predicted = outputs.max(1)
            val_total += batch_y.size(0)
            val_correct += predicted.eq(batch_y).sum().item()

    val_loss = val_loss_sum / len(fashion_test_loader)
    val_acc = val_correct / val_total

    # 记录到TensorBoard / Log to TensorBoard
    fashion_writer.add_scalar('Loss/train', train_loss, epoch)
    fashion_writer.add_scalar('Loss/val', val_loss, epoch)
    fashion_writer.add_scalar('Accuracy/train', train_acc, epoch)
    fashion_writer.add_scalar('Accuracy/val', val_acc, epoch)
    fashion_writer.add_scalars('Loss/compare', {'train': train_loss, 'val': val_loss}, epoch)
    fashion_writer.add_scalars('Accuracy/compare', {'train': train_acc, 'val': val_acc}, epoch)

    # 记录卷积核图像 / Log convolutional filter images
    if epoch == 0:
        filters = fashion_model.features[0].weight.data
        filter_grid = make_grid(filters, nrow=4, normalize=True)
        fashion_writer.add_image('Filters/conv1', filter_grid, epoch)

    # 记录学习率 / Log learning rate
    current_lr = fashion_optimizer.param_groups[0]['lr']
    fashion_writer.add_scalar('Learning_Rate/Adam', current_lr, epoch)

    print(f"Epoch {epoch+1}/5 | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

fashion_writer.close()
print(f"\nFashion-MNIST训练日志保存到: {fashion_logdir}")

## 8. 对比不同模型配置

使用TensorBoard对比不同超参数配置的训练效果。每个配置使用独立的日志目录。

In [ ]:
# 定义不同的配置 / Define different configurations
configs = [
    {'name': 'shallow', 'n_hidden': 1, 'n_neurons': 30},
    {'name': 'deep', 'n_hidden': 3, 'n_neurons': 30},
    {'name': 'wide', 'n_hidden': 1, 'n_neurons': 100},
]

# 训练每个配置 / Train each configuration
for config in configs:
    print(f"\n训练配置: {config['name']}")
    print("=" * 40)

    # 创建模型 / Create model
    model, optimizer, criterion = create_model(
        n_hidden=config['n_hidden'],
        n_neurons=config['n_neurons']
    )

    # 创建独立的日志目录 / Create separate log directory
    logdir = get_run_logdir(config['name'])
    writer = SummaryWriter(log_dir=logdir)

    # 训练 / Train
    history = train_with_tensorboard(
        model, optimizer, criterion,
        train_loader, valid_loader,
        writer,
        epochs=20,
        log_histograms=True,
        log_graph=False
    )

    # 评估 / Evaluate
    model.eval()
    with torch.no_grad():
        test_preds = model(X_test_tensor.to(device))
        test_mse = criterion(test_preds, y_test_tensor.to(device)).item()
        test_mae = nn.L1Loss()(test_preds, y_test_tensor.to(device)).item()

    # 记录最终测试结果 / Log final test results
    writer.add_text('Results/test', f'MSE: {test_mse:.4f}, MAE: {test_mae:.4f}', 0)
    writer.close()

    print(f"测试MSE: {test_mse:.4f}, 测试MAE: {test_mae:.4f}")
    print(f"日志保存到: {logdir}")

## 9. 使用学习率调度器与TensorBoard

PyTorch的学习率调度器(如 `StepLR`, `ReduceLROnPlateau` 等)可以动态调整学习率，
我们可以将学习率变化记录到TensorBoard中。

In [ ]:
# 使用学习率调度器 / Use learning rate scheduler
scheduler_model, scheduler_optimizer, scheduler_criterion = create_model(
    n_hidden=2, n_neurons=50, lr=0.1
)

# 创建StepLR调度器 / Create StepLR scheduler
scheduler = optim.lr_scheduler.StepLR(scheduler_optimizer, step_size=10, gamma=0.5)

scheduler_logdir = get_run_logdir("lr_scheduler")
scheduler_writer = SummaryWriter(log_dir=scheduler_logdir)

# 训练 / Train
for epoch in range(30):
    scheduler_model.train()
    epoch_loss = 0.0
    epoch_mae = 0.0
    n_batches = 0

    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        scheduler_optimizer.zero_grad()
        outputs = scheduler_model(batch_X)
        loss = scheduler_criterion(outputs, batch_y)
        loss.backward()
        scheduler_optimizer.step()
        epoch_loss += loss.item()
        epoch_mae += nn.L1Loss()(outputs, batch_y).item()
        n_batches += 1

    # 调整学习率 / Adjust learning rate
    scheduler.step()

    # 验证 / Validate
    scheduler_model.eval()
    val_loss_sum = 0.0
    val_mae_sum = 0.0
    val_n = 0
    with torch.no_grad():
        for batch_X, batch_y in valid_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = scheduler_model(batch_X)
            val_loss_sum += scheduler_criterion(outputs, batch_y).item() * batch_X.size(0)
            val_mae_sum += nn.L1Loss()(outputs, batch_y).item() * batch_X.size(0)
            val_n += batch_X.size(0)

    train_loss = epoch_loss / n_batches
    val_loss = val_loss_sum / val_n
    current_lr = scheduler_optimizer.param_groups[0]['lr']

    # 记录到TensorBoard / Log to TensorBoard
    scheduler_writer.add_scalar('Loss/train', train_loss, epoch)
    scheduler_writer.add_scalar('Loss/val', val_loss, epoch)
    scheduler_writer.add_scalar('Learning_Rate/SGD', current_lr, epoch)

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:3d}/30 | LR: {current_lr:.6f} | "
              f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

scheduler_writer.close()
print(f"\n学习率调度器日志保存到: {scheduler_logdir}")

## 10. 启动TensorBoard

### 方式一：命令行启动

在终端中运行：
```bash
tensorboard --logdir=./my_logs_pytorch --port=6006
```

然后在浏览器中访问 `http://localhost:6006`

### 方式二：在Jupyter中启动

注意：TensorBoard的启动方式与框架无关，PyTorch和TensorFlow生成的日志文件格式相同，
都可以用同一个TensorBoard服务查看。这就是TensorBoard **框架无关性** 的体现！

In [ ]:
# 在Jupyter中加载TensorBoard扩展 / Load TensorBoard extension in Jupyter
try:
    %load_ext tensorboard
    print("TensorBoard扩展已加载 / TensorBoard extension loaded")
except Exception:
    print("无法加载TensorBoard扩展，请使用命令行启动 / Cannot load extension, use CLI instead")

In [ ]:
# 启动TensorBoard（在Jupyter中）/ Launch TensorBoard in Jupyter
# 取消注释下面的行以启动 / Uncomment the line below to launch
# %tensorboard --logdir ./my_logs_pytorch

## 11. TF vs PyTorch 对照

TensorBoard 是一个 **框架无关** 的可视化工具，PyTorch 和 TensorFlow 共享同一个 TensorBoard 前端。
下表展示了两者在使用 TensorBoard 时的对照关系。

### 核心API对照

| 功能 | TensorFlow/Keras | PyTorch |
|------|-----------------|----------|
| **写入器/回调** | `keras.callbacks.TensorBoard` | `torch.utils.tensorboard.SummaryWriter` |
| **自动记录训练** | 回调自动记录 loss/metrics | 需要手动在训练循环中调用 `add_scalar` |
| **记录标量** | `tf.summary.scalar(name, value, step)` | `writer.add_scalar(tag, value, step)` |
| **记录多个标量** | 多次调用 `tf.summary.scalar` | `writer.add_scalars(main_tag, dict, step)` |
| **记录直方图** | `tf.summary.histogram(name, values, step)` | `writer.add_histogram(tag, values, step)` |
| **记录模型图** | 回调自动记录 (`write_graph=True`) | `writer.add_graph(model, input)` |
| **记录图像** | `tf.summary.image(name, tensor, step)` | `writer.add_image(tag, img_tensor, step)` |
| **记录文本** | `tf.summary.text(name, text, step)` | `writer.add_text(tag, text, step)` |
| **创建写入器** | `tf.summary.create_file_writer(logdir)` | `SummaryWriter(log_dir=logdir)` |
| **刷新** | `writer.flush()` | `writer.flush()` |
| **关闭** | `writer.close()` | `writer.close()` |

### 使用模式对照

| 方面 | TensorFlow/Keras | PyTorch |
|------|-----------------|----------|
| **集成方式** | 声明式回调，`model.fit()` 自动调用 | 命令式，需在训练循环中手动调用 |
| **权重直方图** | `histogram_freq=1` 自动记录 | 遍历 `model.named_parameters()` 手动记录 |
| **学习率记录** | 自定义回调或 `tf.summary.scalar` | `optimizer.param_groups[0]['lr']` + `add_scalar` |
| **超参数搜索** | `tensorboard.plugins.hparams` | 无官方插件，使用 `add_text` 或 `add_hparams` |
| **日志格式** | 事件文件 (`.v2`) | 事件文件 (相同格式) |
| **启动方式** | 完全相同 | 完全相同 |

### 代码对照示例

```python
# ===== TensorFlow / Keras =====
tensorboard_cb = keras.callbacks.TensorBoard(
    log_dir=logdir,
    histogram_freq=1,
    write_graph=True,
)
model.fit(X_train, y_train, callbacks=[tensorboard_cb])

# ===== PyTorch =====
writer = SummaryWriter(log_dir=logdir)
for epoch in range(epochs):
    # ... training code ...
    writer.add_scalar('Loss/train', train_loss, epoch)
    writer.add_scalar('Loss/val', val_loss, epoch)
    for name, param in model.named_parameters():
        writer.add_histogram(f'Weights/{name}', param, epoch)
writer.close()
```

### 关键结论

1. **TensorBoard是框架无关的**：PyTorch和TensorFlow生成的事件文件格式完全相同，可以混合放在同一目录下用同一个TensorBoard实例查看
2. **PyTorch更手动但更灵活**：Keras的回调是自动的，PyTorch需要在训练循环中显式调用，但这赋予了更大的控制力
3. **add_hparams**：PyTorch的SummaryWriter也支持 `add_hparams` 方法来记录超参数
4. **共享前端**：无论用哪个框架生成日志，都使用完全相同的 `tensorboard --logdir` 命令启动

In [ ]:
# 演示 add_hparams 记录超参数 / Demo add_hparams for hyperparameter logging
hparam_logdir = get_run_logdir("hparams_demo")
hparam_writer = SummaryWriter(log_dir=hparam_logdir)

# 模拟超参数搜索结果 / Simulate hyperparameter search results
hparam_experiments = [
    {'n_hidden': 1, 'n_neurons': 30, 'lr': 0.01},
    {'n_hidden': 2, 'n_neurons': 50, 'lr': 0.01},
    {'n_hidden': 2, 'n_neurons': 100, 'lr': 0.005},
    {'n_hidden': 3, 'n_neurons': 50, 'lr': 0.01},
]

# 模拟结果 / Simulated results
simulated_results = [
    {'mse': 0.5234, 'mae': 0.5123},
    {'mse': 0.4567, 'mae': 0.4789},
    {'mse': 0.4123, 'mae': 0.4456},
    {'mse': 0.4890, 'mae': 0.4901},
]

for i, (hparams, metrics) in enumerate(zip(hparam_experiments, simulated_results)):
    # 每个实验创建独立的writer / Create separate writer per experiment
    exp_logdir = os.path.join(hparam_logdir, f"exp_{i}")
    exp_writer = SummaryWriter(log_dir=exp_logdir)

    # 记录超参数和指标 / Log hyperparameters and metrics
    exp_writer.add_hparams(
        hparam_dict=hparams,
        metric_dict=metrics,
        hparam_domain_discrete={
            'n_hidden': [1, 2, 3],
            'n_neurons': [30, 50, 100],
        }
    )

    exp_writer.close()
    print(f"实验 {i}: hparams={hparams}, metrics={metrics}")

hparam_writer.close()
print(f"\n超参数实验日志保存到: {hparam_logdir}")
print("在TensorBoard的HPARAMS标签页中可以查看超参数对比。")

## 12. 混合框架日志演示

以下代码演示如何将PyTorch和TensorFlow的日志放在同一目录下，
用同一个TensorBoard实例查看。这体现了TensorBoard的框架无关性。

In [ ]:
# 混合框架日志演示 / Mixed framework logging demo
# 将PyTorch和TensorFlow的日志放在同一目录 / Put PyTorch and TF logs in same directory
mixed_logdir = os.path.join(os.curdir, "my_logs_mixed")
os.makedirs(mixed_logdir, exist_ok=True)

# PyTorch日志 / PyTorch logs
pytorch_writer = SummaryWriter(log_dir=os.path.join(mixed_logdir, "pytorch_run"))
for i in range(50):
    pytorch_writer.add_scalar('Loss/train', 1.0 / (i + 1) + np.random.normal(0, 0.02), i)
    pytorch_writer.add_scalar('Loss/val', 1.1 / (i + 1) + np.random.normal(0, 0.03), i)
pytorch_writer.close()

# TensorFlow日志 (如果可用) / TensorFlow logs (if available)
try:
    import tensorflow as tf
    tf_writer = tf.summary.create_file_writer(os.path.join(mixed_logdir, "tensorflow_run"))
    with tf_writer.as_default():
        for i in range(50):
            tf.summary.scalar('Loss/train', 1.05 / (i + 1) + np.random.normal(0, 0.02), step=i)
            tf.summary.scalar('Loss/val', 1.15 / (i + 1) + np.random.normal(0, 0.03), step=i)
    tf_writer.close()
    print("PyTorch和TensorFlow日志均已写入混合目录！")
    print("使用 tensorboard --logdir=./my_logs_mixed 可以同时查看两个框架的训练曲线。")
except ImportError:
    print("TensorFlow未安装，仅生成了PyTorch日志。")
    print("但原理相同：PyTorch和TF的日志文件格式完全兼容！")

print(f"\n混合日志目录: {mixed_logdir}")

## 13. 清理日志文件

In [ ]:

# 列出所有日志目录 / List all log directories
print("已创建的PyTorch日志目录:")
if os.path.exists(root_logdir):
    for item in sorted(os.listdir(root_logdir)):
        item_path = os.path.join(root_logdir, item)
        if os.path.isdir(item_path):
            size = sum(
                os.path.getsize(os.path.join(dirpath, f))
                for dirpath, dirnames, filenames in os.walk(item_path)
                for f in filenames
            ) / 1024
            print(f"  {item} ({size:.1f} KB)")

# 取消注释以下代码来清理日志 / Uncomment to clean up logs
# if os.path.exists(root_logdir):
#     shutil.rmtree(root_logdir)
#     print("\n已清理所有PyTorch日志文件")

## 小结

### SummaryWriter 主要方法

| 方法 | 参数 | 说明 |
|------|------|------|
| `add_scalar` | tag, scalar_value, step | 记录标量(损失、精度等) |
| `add_scalars` | main_tag, tag_dict, step | 在同一图中记录多个标量 |
| `add_histogram` | tag, values, step | 记录直方图(权重、梯度) |
| `add_graph` | model, input_to_model | 记录模型计算图 |
| `add_image` | tag, img_tensor, step | 记录图像 |
| `add_images` | tag, img_tensor_batch, step | 记录多张图像 |
| `add_text` | tag, text_string, step | 记录文本 |
| `add_hparams` | hparam_dict, metric_dict | 记录超参数和指标 |
| `flush` | - | 将缓冲区写入磁盘 |
| `close` | - | 关闭写入器 |

### SummaryWriter 构造函数参数

| 参数 | 默认值 | 说明 |
|------|--------|------|
| `log_dir` | `runs/CURRENT_DATETIME_HOSTNAME` | 日志保存目录 |
| `comment` | `''` | 日志目录后缀注释 |
| `purge_step` | `None` | 从指定步骤开始清除日志 |
| `max_queue` | `10` | 队列中最大事件数 |
| `flush_secs` | `120` | 自动刷新间隔(秒) |
| `filename_suffix` | `''` | 事件文件名后缀 |

### 最佳实践

1. **命名规范**: 为每个实验使用描述性名称，如 `add_scalar('Loss/train', ...)` 而不是 `add_scalar('loss', ...)`
2. **版本控制**: 在日志目录名中包含时间戳和实验名称
3. **定期flush**: 长时间训练时定期调用 `writer.flush()` 确保数据写入磁盘
4. **及时关闭**: 训练结束后调用 `writer.close()` 释放资源
5. **使用with语句**: `with SummaryWriter(...) as writer:` 可以自动关闭
6. **混合框架**: PyTorch和TF的日志可以放在同一目录下，用同一个TensorBoard查看

### 常用命令

```bash
# 基本启动 / Basic launch
tensorboard --logdir=./my_logs_pytorch

# 指定端口 / Specify port
tensorboard --logdir=./my_logs_pytorch --port=6007

# 绑定所有网络接口（远程访问）/ Bind all interfaces (remote access)
tensorboard --logdir=./my_logs_pytorch --host=0.0.0.0

# 同时查看多个日志目录 / View multiple log directories
tensorboard --logdir_spec=pytorch:./my_logs_pytorch,tf:./my_logs
```

---

## 练习

### 练习1: 自定义训练循环中的TensorBoard记录

修改上面的训练函数，添加以下功能：
1. 每个 batch 记录一次 loss（使用 `add_scalar` + `global_step` 跟踪总 batch 数）
2. 每5个 epoch 记录一次卷积层权重的直方图
3. 使用 `add_text` 记录每个 epoch 的超参数配置摘要

提示：
```python
global_step = 0
for epoch in range(epochs):
    for batch_X, batch_y in train_loader:
        # ... training code ...
        writer.add_scalar('Loss/batch', loss.item(), global_step)
        global_step += 1
```

### 练习2: 使用TensorBoard对比不同优化器

使用相同模型架构，分别用 SGD、Adam、RMSprop 训练，将日志保存到不同子目录中。
在 TensorBoard 中对比它们的训练曲线。

要求：
- 记录每个优化器的学习率变化
- 记录每个优化器的梯度范数（`param.grad.norm()`）
- 使用 `add_scalars` 在同一图中对比三种优化器的验证损失

### 练习3: 使用Fashion-MNIST记录预测结果图像

训练一个Fashion-MNIST分类器后，从测试集中选取一些样本，
将模型预测结果和真实标签用 `add_image` 和 `add_text` 记录到 TensorBoard。

提示：
```python
# 创建包含预测结果的图像 / Create image with prediction results
import torchvision
wrong_preds = []
wrong_labels = []
for img, pred, true in zip(images, predictions, labels):
    if pred != true:
        wrong_preds.append(img)
wrong_grid = torchvision.utils.make_grid(wrong_preds[:16], nrow=4)
writer.add_image('Predictions/wrong', wrong_grid, epoch)
```